In [ ]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

model_name = "gpt-oss:20b"
chat_generator = OllamaChatGenerator(model=model_name,
    url = "",
    timeout=300,
    generation_kwargs={
        "num_predict": 3000,
        "temperature": 0.2,
    },
)

In [2]:
# =========================
# IMPORTS
# =========================
import os
import json
import requests
from dataclasses import dataclass, field
from typing import Callable, List

from dotenv import load_dotenv
#from bs4 import BeautifulSoup
from urllib.parse import urljoin

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

load_dotenv()

True

In [3]:
# =========================
# TAVILY WEB SEARCH
# =========================
class TavilyWebSearch:
    def __init__(self, api_key: str, top_k: int = 5):
        self.api_key = api_key
        self.top_k = top_k

    def run(self, query: str):
        payload = {
            "api_key": self.api_key,
            "query": query,
            "max_results": self.top_k,
            "include_domains": [
                "arxiv.org",
                "openreview.net",
                "aclanthology.org",
                "ieee.org",
            ],
        }

        r = requests.post(
            "https://api.tavily.com/search",
            json=payload,
            timeout=15,
        )

        data = r.json()

        return {
            "papers": [
                {"title": x["title"], "url": x["url"]}
                for x in data.get("results", [])
            ]
        }


tavily = TavilyWebSearch(api_key=os.environ["TAVILY_API_KEY"])


def web_search(query: str) -> dict:
    return tavily.run(query)



In [4]:
# =========================
# DOWNLOAD PDFs
# =========================
def download_pdfs(urls: list) -> list:
    save_dir = os.path.join(os.getcwd(), "downloads")
    os.makedirs(save_dir, exist_ok=True)

    downloaded = []

    for url in urls:
        try:
            print(f"[DOWNLOAD] {url}")

            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                continue

            filename = url.split("/")[-1]
            if not filename.endswith(".pdf"):
                filename = filename[:60] + ".pdf"

            path = os.path.join(save_dir, filename)

            with open(path, "wb") as f:
                f.write(r.content)

            downloaded.append(path)
            print(f"[SAVED] {path}")

        except Exception as e:
            print(f"[FAIL] {url}: {e}")

    return downloaded



In [5]:
from dataclasses import dataclass, field
from typing import Callable, List

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker


@dataclass
class ToolCallingAgent:
    name: str = "ToolCallingAgent"
    llm: object = None
    instructions = """
You are an autonomous research agent.

GOAL:
Find academic papers and download their PDFs.

WORKFLOW:
1. Use web_search to find papers
2. Inspect returned URLs carefully
3. Convert arXiv /abs/ links to /pdf/
4. Directly call download_pdfs with final PDF URLs
5. Ensure at least 3 PDFs are downloaded
6. Retry search if needed

IMPORTANT:
- Do NOT use intermediate tools
- Do NOT assume extraction tools exist
- You are responsible for selecting final PDF URLs
"""
    functions: list[Callable] = field(default_factory=list)

    def __post_init__(self):

        self._system_message = ChatMessage.from_system(self.instructions)

        self.tools = [
            create_tool_from_function(fun)
            for fun in self.functions
        ]

        self._tool_invoker = ToolInvoker(
            tools=self.tools,
            raise_on_failure=False
        )

    def run(self, messages: list[ChatMessage], max_steps: int = 6):

        history = [self._system_message] + messages

        for step in range(max_steps):

            # 1. LLM call
            agent_message = self.llm.run(
                messages=history,
                tools=self.tools
            )["replies"][0]

            history.append(agent_message)

            # 2. STOP if no tool calls
            if not agent_message.tool_calls:
                break

            # 3. Execute tools
            tool_results = self._tool_invoker.run(
                messages=[agent_message]
            )["tool_messages"]

            history.extend(tool_results)

        return history

In [6]:
agent = ToolCallingAgent(
    llm=chat_generator,
    functions=[web_search, download_pdfs]
)

In [7]:
from haystack.dataclasses import ChatMessage

messages = [
    ChatMessage.from_user("Find Paper about Agentic AI")
]

In [8]:
response = agent.run(messages)

[DOWNLOAD] https://arxiv.org/pdf/2510.25445.pdf
[SAVED] c:\Users\olena\Olena_Tsvietkova\Forschungsmethoden\dat_project\notebooks\downloads\2510.25445.pdf
[DOWNLOAD] https://arxiv.org/pdf/2503.00237v1.pdf
[SAVED] c:\Users\olena\Olena_Tsvietkova\Forschungsmethoden\dat_project\notebooks\downloads\2503.00237v1.pdf
[DOWNLOAD] https://arxiv.org/pdf/2506.02153.pdf
[SAVED] c:\Users\olena\Olena_Tsvietkova\Forschungsmethoden\dat_project\notebooks\downloads\2506.02153.pdf


In [9]:
for msg in response:
    print(msg)


ChatMessage(_role=<ChatRole.SYSTEM: 'system'>, _content=[TextContent(text='\nYou are an autonomous research agent.\n\nGOAL:\nFind academic papers and download their PDFs.\n\nWORKFLOW:\n1. Use web_search to find papers\n2. Inspect returned URLs carefully\n3. Convert arXiv /abs/ links to /pdf/\n4. Directly call download_pdfs with final PDF URLs\n5. Ensure at least 3 PDFs are downloaded\n6. Retry search if needed\n\nIMPORTANT:\n- Do NOT use intermediate tools\n- Do NOT assume extraction tools exist\n- You are responsible for selecting final PDF URLs\n')], _name=None, _meta={})
ChatMessage(_role=<ChatRole.USER: 'user'>, _content=[TextContent(text='Find Paper about Agentic AI')], _name=None, _meta={})
ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[ReasoningContent(reasoning_text='The user wants a paper about Agentic AI. We need to find academic papers and download PDFs. We must use web_search to find papers. Then inspect returned URLs carefully. Convert arXiv /abs/ links to 